# Specificity: a true-negative done right

**Utility:** a good viral detector must *not* cry wolf. The manuscript applied
ViralScan to two COVID-era clinical libraries and found **zero** SARS-CoV-2 UMIs —
cross-confirmed by an independent STARsolo run. This notebook shows the two things
that make a true-negative trustworthy: catching a **chemistry mismatch** before it
produces garbage, and reading a **zero-count** summary correctly.

## 1 — catch the chemistry mismatch (runnable)

The COVID libraries used GEM-X 5′ chemistry; only **0.4%** of R1 barcodes matched the
10x v3 whitelist. Running as v3 would have silently produced a near-empty matrix.
`whitelist_match_rate` samples R1 and reports the fraction of barcodes in the
whitelist — a fast preflight. Here is a synthetic demo of a match vs a mismatch.

In [ ]:
import tempfile
from pathlib import Path
from viralscan.whitelist_preflight import whitelist_match_rate

tmp = Path(tempfile.mkdtemp())
CB_LEN = 16
whitelist = {'AAAAAAAAAAAAAAAA', 'CCCCCCCCCCCCCCCC', 'GGGGGGGGGGGGGGGG'}

def make_r1(path, barcodes):
    with open(path, 'w') as fh:
        for i, bc in enumerate(barcodes):
            fh.write(f'@read{i}\n{bc}NNNNNNNN\n+\n{"I" * (len(bc) + 8)}\n')

# Correct chemistry: barcodes are in the whitelist.
good = tmp / 'good_R1.fastq'
make_r1(good, ['AAAAAAAAAAAAAAAA', 'CCCCCCCCCCCCCCCC', 'GGGGGGGGGGGGGGGG'] * 20)
rate_good, n = whitelist_match_rate(str(good), whitelist, CB_LEN)

# Wrong chemistry: barcodes are a different universe (nothing matches).
bad = tmp / 'bad_R1.fastq'
make_r1(bad, ['TTTTTTTTTTTTTTTT', 'ATATATATATATATAT', 'CGCGCGCGCGCGCGCG'] * 20)
rate_bad, _ = whitelist_match_rate(str(bad), whitelist, CB_LEN)

print(f'correct chemistry match rate: {rate_good:.0%}  (n={n})  -> proceed')
print(f'wrong chemistry match rate  : {rate_bad:.0%}         -> STOP: wrong --technology')

In a real run this is one command; a rate far below `--min-match-rate` (default 0.5)
means you picked the wrong chemistry or whitelist:

```bash
viralscan check-whitelist -s1 R1.fastq.gz -w 10x_v3_whitelist.txt -x 10xv3
# match rate 0.004 -> GEM-X library; use the correct whitelist / raw-barcode universe.
```

## 2 — read a zero-count summary correctly (runnable)

When a target virus is truly absent, its row is simply absent (or zero) in
`viral_summary.tsv`. The trust comes from the *negative control accession*: the paper
included SARS-CoV-1 (`NC_004718.3`) alongside SARS-CoV-2 — both zero. Here we simulate
a matrix with the viral columns genuinely empty and confirm ViralScan reports nothing.

In [ ]:
import numpy as np, anndata as ad, scipy.sparse as sp
from viralscan.scripts.detection import compute_stats

# 4 cells, 3 genes: 2 viral columns are all-zero (virus absent), 1 host column real.
X = np.array([[0, 0, 1200], [0, 0, 900], [0, 0, 1500], [0, 0, 800]], dtype=float)
adata = ad.AnnData(sp.csr_matrix(X))
adata.obs_names = [f'bc{i}' for i in range(4)]
adata.var_names = ['SARS_CoV_2', 'SARS_CoV_1', 'HOST_ACTB']

stats, per_cell = compute_stats(
    adata, {}, {'SARS-CoV-2': ['SARS_CoV_2'], 'SARS-CoV-1': ['SARS_CoV_1']}, [])
print('viruses with any UMI:', {k: v['total_umi'] for k, v in stats.items()})
print('infected cells in per-cell table:', len(per_cell))
assert all(v['total_umi'] == 0 for v in stats.values())
print('OK — no false-positive viral calls when the virus is absent.')

## 3 — cell-calling concordance (why the negative is credible)

A zero viral count is only meaningful if the cells are real. On the COVID sample,
ViralScan's cell calls agreed closely with CellRanger, so the zero is a zero over
*genuine cells*, not an empty matrix (manuscript, *ViralScan detects no SARS-CoV-2*):

| Method | Cells called (LUM-SJ-x213-g) | CellRanger overlap |
|--------|------------------------------|--------------------|
| CellRanger (reference) | 28,922 | 100% (anchor) |
| ViralScan emptyDrops | 30,849 | 81.4% (Jaccard 0.65) |
| STARsolo EmptyDrops_CR | 19,920 | subset of CellRanger |

You can also feed CellRanger's own barcodes as the denominator:
`--cell-calling external --called-cells-file cellranger_barcodes.tsv`.

## Summary

- Preflight chemistry with `check-whitelist`; a low match rate means wrong chemistry.
- Include a **negative-control accession** (e.g. a taxonomically adjacent virus).
- Confirm cell-calling concordance so the zero is over real cells.
- ViralScan produced no false-positive SARS-CoV-2 calls — supporting clinical
  specificity — but see the host-homology caveat in **qc_and_read_evidence** for
  library types (whole-blood, high-intron) that need an orthogonal genome control.